# Entity ***Ticks***
+ Layer bronze
+ Priority 2
---
### Imports

In [0]:
%run ../00_functions/medallion_functions

In [0]:
import requests
import time
from pyspark.sql.functions import current_timestamp

### Parameters

In [0]:
dbutils.widgets.text("layer", "bronze")
dbutils.widgets.text("entity_name", "ticks")
dbutils.widgets.text("load_pattern", "1")
dbutils.widgets.text("SUBGRAPH_URL", "https://gateway.thegraph.com/api/a5bbc98aac75dac555f9aba8747c7e2e/subgraphs/id/5zvR82QoaXYFyDEKLZ9t6v9adgnptxYpKpSbxtgVENFV")

In [0]:
layer = dbutils.widgets.get("layer")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")
SUBGRAPH_URL = dbutils.widgets.get("SUBGRAPH_URL")

### Variables

In [0]:
PAGE_SIZE = 1000
BATCH_SIZE = 200
ticks_list = []
query_variables= {
    "batch_pool_ids": [],
    "skip": 0
}

In [0]:
ticks_query = """
query TicksQueryBatch($batch_pool_ids: [String!]!, $skip: Int!) {
  ticks(
    where: {
      pool_in: $batch_pool_ids,
      liquidityNet_not: "0",
      liquidityGross_not: "0"
    },
    first: 1000,
    skip: $skip, 
  ){
    id
    poolAddress
    tickIdx
    pool
    volumeToken0
    volumeToken1
    volumeUSD
    price0
    price1
    liquidityGross
    liquidityNet
    feesUSD
    collectedFeesToken0
    collectedFeesToken1
    collectedFeesUSD
    createdAtTimestamp
    createdAtBlockNumber
  }
}
"""

### Filtering relevant pools
+ Ticks could overload the graph queries threshold since within a single pools could exists a large amount of ticks.

In [0]:
filtered_bz_pools_df = spark.read.table("uniswap.bronze.pools")\
      .filter("""liquidity != '0' 
              AND CAST(totalValueLockedUSD AS DOUBLE) > 10000
              AND CAST(volumeUSD AS DOUBLE) > 1000
              AND CAST(txCount AS INT) > 10
              AND feeTier IN ('500', '3000', '10000', '100000')
              """)

### Execution

In [0]:
pool_ids_list = [row.id for row in filtered_bz_pools_df.select("id").collect()]

In [0]:
for i in range(0, len(pool_ids_list), BATCH_SIZE):
    batch_pools = pool_ids_list[i:i+BATCH_SIZE]
    query_variables["batch_pool_ids"] = batch_pools
    query_variables["skip"] = 0
    print(f"*INFO*: Loading ticks for {len(batch_pools)} pools.")
    while True:
        response = requests.post(SUBGRAPH_URL, json={"query": ticks_query, "variables": query_variables})
        ticks_data = response.json()["data"][entity_name]

        if "errors" in response.json():
            raise Exception(response.json()["errors"])
        
        ticks_list.extend(ticks_data)

        time.sleep(0.5)

        rows_loaded = len(ticks_data)
        print(f"*INFO*: Number of rows loaded: {rows_loaded}.")
        
        if rows_loaded < PAGE_SIZE:
            break
           
        query_variables["skip"] += PAGE_SIZE
        

In [0]:
ticks_df = spark.createDataFrame(ticks_list)
ticks_df = ticks_df.withColumn("load_timestamp", current_timestamp())

In [0]:
display(ticks_df.count())

### Save & exit

In [0]:
load_result = load_entity(ticks_df,
                        entity_name,
                        load_pattern,
                        layer
                        #,
                        )

In [0]:
dbutils.notebook.exit(load_result)